# Lesson 24: Factor Models
**Module 4: ML Regression**

Interactive notebook for CAPM and Fama-French factor analysis.

## Learning Objectives
- Estimate CAPM beta and alpha
- Apply multi-factor regression
- Interpret factor loadings
- Perform risk attribution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
np.random.seed(42)

# Simulate factor returns (252 trading days)
n = 252
mkt_rf = np.random.randn(n) * 0.01  # Market excess return
smb = np.random.randn(n) * 0.005    # Size factor
hml = np.random.randn(n) * 0.005    # Value factor

# Simulate stock returns with known factor exposures
true_alpha = 0.0002  # 2 bps daily alpha
true_beta = 1.2
true_smb = 0.3
true_hml = -0.2

stock_return = (true_alpha + true_beta * mkt_rf + 
                true_smb * smb + true_hml * hml + 
                np.random.randn(n) * 0.005)

df = pd.DataFrame({
    'stock': stock_return,
    'mkt_rf': mkt_rf,
    'smb': smb,
    'hml': hml
})
print('Factor data ready!')
df.head()

## 1. CAPM Regression

In [ ]:
# Single factor model
X_capm = sm.add_constant(df['mkt_rf'])
capm = sm.OLS(df['stock'], X_capm).fit()
print(capm.summary().tables[1])
print(f'\nBeta: {capm.params["mkt_rf"]:.3f}')
print(f'Alpha (daily): {capm.params["const"]:.6f}')

In [ ]:
# Scatter plot with regression line
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['mkt_rf'], df['stock'], alpha=0.5)
x_line = np.linspace(df['mkt_rf'].min(), df['mkt_rf'].max(), 100)
ax.plot(x_line, capm.params['const'] + capm.params['mkt_rf'] * x_line, 
        color='red', linewidth=2, label=f'Beta={capm.params["mkt_rf"]:.2f}')
ax.set_xlabel('Market Excess Return')
ax.set_ylabel('Stock Return')
ax.set_title('CAPM Regression')
ax.legend()
plt.show()

## 2. Fama-French Three-Factor

In [ ]:
# Three-factor model
X_ff3 = sm.add_constant(df[['mkt_rf', 'smb', 'hml']])
ff3 = sm.OLS(df['stock'], X_ff3).fit()
print(ff3.summary().tables[1])
print(f'\nR-squared: {ff3.rsquared:.3f} (vs CAPM: {capm.rsquared:.3f})')

## 3. Rolling Betas

In [ ]:
# Calculate 60-day rolling beta
window = 60
rolling_beta = df['stock'].rolling(window).cov(df['mkt_rf']) / df['mkt_rf'].rolling(window).var()

plt.figure(figsize=(12, 5))
plt.plot(rolling_beta)
plt.axhline(y=true_beta, color='r', linestyle='--', label=f'True beta={true_beta}')
plt.title('60-Day Rolling Beta')
plt.ylabel('Beta')
plt.legend()
plt.show()

## Exercise: Risk Attribution
Decompose total variance into factor and idiosyncratic.

In [ ]:
# Exercise: Calculate risk decomposition
# 1. Fitted values = factor risk
# 2. Residuals = idiosyncratic risk
# 3. What % of variance is explained by factors?

# YOUR CODE HERE

---
## Solution
<details>
<summary>Click to reveal</summary>

```python
# Risk decomposition
fitted_values = ff3.fittedvalues
residuals = ff3.resid

total_var = df['stock'].var()
factor_var = fitted_values.var()
idio_var = residuals.var()

print(f'Total variance: {total_var:.6f}')
print(f'Factor variance: {factor_var:.6f} ({100*factor_var/total_var:.1f}%)')
print(f'Idiosyncratic variance: {idio_var:.6f} ({100*idio_var/total_var:.1f}%)')
print(f'\nR-squared check: {ff3.rsquared:.3f}')
```
</details>